# 04 · TabPFN-3.5 vs six classic learners and the market itself (CPU only)

The protocol is fixed in [`PREREG.md`](../PREREG.md), which was written before any test fold was run.
- Weekly walk-forward test blocks.
- Every model trains only on markets that closed before the block starts.
- **Policy A:** every model sees the same 5,000 most recent rows.
- Classic learners are tuned on a validation slice (defaults + 20 random configurations). TabPFN is not tuned.

`make benchmark` reruns everything (a few hours on an 8-core CPU). This notebook reads the cached
results in `results/`.

In [ ]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from tabtrader import plots
plots.style()
pd.set_option("display.width", 140, "display.max_columns", 30)

In [ ]:
S = json.loads((ROOT / "results" / "summary.json").read_text())
def board(market, policy="policy_A"):
    L = S[market][policy]
    rows = []
    for m, r in L["models"].items():
        v, mk = r.get("vs_ref"), r.get("vs_market")
        rows.append({"model": m, "log loss": r["logloss"], "Brier": r["brier"], "AUC": r["auc"], "ECE": r["ece"],
                     "Δ vs TabPFN-3.5": None if not v else v["mean"],
                     "95% CI": None if not v else f"[{v['lo']:+.4f}, {v['hi']:+.4f}]",
                     "BH": None if not v else v.get("bh_pass"),
                     "Δ vs market": None if not mk else mk["mean"],
                     "fit s": r["cpu"].get("fit_s_mean"), "tune s": r["cpu"].get("search_s_mean"),
                     "1 market s": r["cpu"].get("single_market_s")})
    return pd.DataFrame(rows).sort_values("log loss").set_index("model")
print(S["15m"]["policy_A"]["n_rows"], "test rows over", S["15m"]["policy_A"]["n_days"], "days (15-minute)")
board("15m").round(4)

In [ ]:
print(S["1h"]["policy_A"]["n_rows"], "test rows over", S["1h"]["policy_A"]["n_days"], "days (hourly)")
board("1h").round(4)

### Paired difference in log loss vs TabPFN-3.5 (right of zero = the model is worse than TabPFN)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=False)
for ax, mk in zip(axes, ["15m", "1h"]):
    L = S[mk]["policy_A"]["models"]
    items = sorted([(m, r["vs_ref"]) for m, r in L.items() if r.get("vs_ref")], key=lambda x: x[1]["mean"])
    for i, (m, v) in enumerate(items):
        ax.plot([v["lo"], v["hi"]], [i, i], color=plots.color(m), lw=2)
        ax.plot(v["mean"], i, "o", color=plots.color(m), ms=7)
    ax.set_yticks(range(len(items)), [m for m, _ in items]); ax.axvline(0, color=plots.INK, lw=1)
    ax.set_title({"15m": "15-minute", "1h": "hourly"}[mk]); ax.set_xlabel("Δ log loss (model − TabPFN-3.5)")
plt.tight_layout()

### Learning curves: log loss vs the number of training rows (library defaults)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, mk in zip(axes, ["15m", "1h"]):
    lc = S[mk]["learning_curve"]
    ns = [int(n) for n in lc]
    models = sorted({m for n in lc for m in lc[n]})
    for m in models:
        ys = [lc[n].get(m, {}).get("logloss") for n in lc]
        if m == "Market price":
            ax.axhline(ys[-1], color=plots.INK, ls="--", lw=1.2, label="market price")
            continue
        hi = m.startswith("TabPFN")
        ax.plot(ns, ys, marker="o", ms=4, color=plots.color(m), lw=2.2 if hi else 1, alpha=1 if hi else .7,
                label=m.replace(" · default", ""))
    ax.set_xscale("log"); ax.set_xlabel("training rows"); ax.set_ylabel("test log loss")
    ax.set_title({"15m": "15-minute", "1h": "hourly"}[mk])
axes[1].legend(fontsize=7.5, loc="upper right"); plt.tight_layout()

### Calibration (reliability) on the test blocks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, mk in zip(axes, ["15m", "1h"]):
    ax.plot([0, 1], [0, 1], color=plots.MUTED, ls="--", lw=1)
    for m in ["Market price", "TabPFN-3.5", "LightGBM · tuned", "Random forest · tuned"]:
        r = S[mk]["policy_A"]["models"].get(m)
        if not r: continue
        rel = pd.DataFrame(r["reliability"])
        ax.plot(rel.p_mean, rel.y_mean, marker="o", ms=4, color=plots.color(m),
                lw=2 if m.startswith("TabPFN") else 1.2, label=f"{m} (ECE {r['ece']:.3f})")
    ax.set_title({"15m": "15-minute", "1h": "hourly"}[mk]); ax.legend(fontsize=7.5)
plt.tight_layout()

### Policy B: the classic learners get *all* history; TabPFN stays at 5,000 rows

In [ ]:
for mk in ["15m", "1h"]:
    if S[mk].get("policy_B"):
        print(mk); display(board(mk, "policy_B")[["log loss", "AUC", "Δ vs TabPFN-3.5", "95% CI", "BH"]].round(4))

### TabPFN-3.5 Thinking (API, time-aware) on the same blocks

In [ ]:
for mk in ["15m", "1h"]:
    th = S[mk].get("thinking")
    if not th: continue
    rows = {m: {"log loss": r["logloss"], "AUC": r["auc"],
                "Thinking − model": (r.get("thinking_minus_model") or {}).get("mean")}
            for m, r in th.items() if isinstance(r, dict) and "logloss" in r}
    print(mk, "blocks", th["blocks"]); display(pd.DataFrame(rows).T.round(4))

### From probabilities to trades

The calibrated-edge rule buys only when the model's calibrated probability beats the ask plus the fee
by a margin chosen on validation. Results are in cents per contract after taker fees,
trade-weighted, with day-block 95% CIs.

In [ ]:
for mk in ["15m", "1h"]:
    T = S[mk]["trading"]
    print(mk)
    display(pd.DataFrame({m: {k: v for k, v in r.items() if k not in ("cum", "margins_c")} for m, r in T.items()}).T)

### Live demo: fit TabPFN-3.5-Fast on CPU and score one market

In [ ]:
import os, time
from tabtrader.data import load_market, MARKETS, blocks, split_block, xy
from tabtrader.models import make_tabpfn
if os.environ.get("TABPFN_TOKEN"):
    df = load_market("15m"); spec = MARKETS["15m"]; a, b = blocks(spec)[-1]
    fit, val, test = split_block(df, spec, a, b, 5000)
    X, y = xy(fit); Xt, yt = xy(test)
    m = make_tabpfn("3.5-fast"); t0 = time.perf_counter(); m.fit(X, y); t_fit = time.perf_counter() - t0
    t0 = time.perf_counter(); p1 = m.predict_proba(Xt[:1])[0, 1]; t_one = time.perf_counter() - t0
    print(f"fit on {len(X):,} rows: {t_fit:.1f}s · one live market scored in {t_one*1000:.0f} ms → P(YES)={p1:.3f}"
          f" vs market {test.p_mid.iloc[0]:.3f}")
else:
    print("set TABPFN_TOKEN (https://ux.priorlabs.ai) to run the live demo")